# Python程序设计（项目实战） · 工单P4.1《多源数据采集系统》
**第6周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 采集率统计面板：实时打印成功率/延迟/丢弃数（简易APM）；示范讲解 |
| **B 组长** | 标准达成 | 断线重连实测通过 |
| **C 操作员** | 基础完成 | 三源贯通+采集率>95% |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：为什么‘共享list.append’有隐患而Queue安全？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 三、任务要求（Make）
**实施步骤**
1. 启动plc_simulator三台PLC
2. ModbusCollector轮询0.5s×3台(÷10还原)
3. read_one退避重试(1→2→4s)

**验收标准**：及格(D)=`三源贯通+采集率>95%` ；优秀(B+)=`断线重连实测通过`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""P4.1 多源工业数据采集系统 —— 三线程架构骨架(已给, 聚焦协议集成与可靠性)
前置: python ../工具/plc_simulator.py  (三台PLC: 5020/5021/5022)
     python ../02-Python编程基础/虚拟产线/mini_mqtt_broker.py
"""
import json
import logging
import queue
import sqlite3
import threading
import time

from pymodbus.client import ModbusTcpClient as Client
import paho.mqtt.client as mqtt

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(threadName)s %(levelname)s %(message)s")
log = logging.getLogger("collector")

PLCS = [("127.0.0.1", 5020), ("127.0.0.1", 5021), ("127.0.0.1", 5022)]
NAMES = ["temp_c", "press_mpa", "vib_mm_s", "rpm", "status"]

### Starter 2：`class ModbusCollector(threading.Thread):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class ModbusCollector(threading.Thread):
    """TODO-1: 轮询三台PLC(0.5s), 读5寄存器÷10还原, 放入Queue
    可靠性: 连接失败退避重试(1s→2s→4s封顶); 2号PLC会随机掉线(实测点)"""

    def __init__(self, q: queue.Queue):
        super().__init__(daemon=True, name="Modbus")
        self.q, self.stop = q, threading.Event()
        self.stat = {"ok": 0, "fail": 0}

    def read_one(self, host, port):
        # === 你的代码: Client(host,port) → read_holding_registers(0,5) ===
        # 返回 dict(含source=f"plc@{port}") 或 None
        raise NotImplementedError

    def run(self):
        backoff = 1
        while not self.stop.is_set():
            got_any = False
            for host, port in PLCS:
                data = self.read_one(host, port)
                if data:
                    self.stat["ok"] += 1
                    self.q.put(data, timeout=1)
                    got_any = True
                else:
                    self.stat["fail"] += 1
            backoff = 1 if got_any else min(backoff * 2, 4)
            time.sleep(0.5 if got_any else backoff)

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - （在此处编写你的代码——依据上方注释/文档字符串）

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`三源贯通+采集率>95%`，优秀(B+)=`断线重连实测通过`

### Starter 3：`class MQTTSubscriber(threading.Thread):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class MQTTSubscriber(threading.Thread):
    """TODO-2: 订阅 device/+/status, 消息JSON解析后放入Queue"""

    def __init__(self, q):
        super().__init__(daemon=True, name="MQTT")
        self.q = q

    def run(self):
        c = mqtt.Client()

        def on_message(_c, _u, msg):
            # === 你的代码: json.loads → dict加source="mqtt" → q.put ===
            pass

        c.on_message = on_message
        c.connect("127.0.0.1", 1883)
        c.subscribe("device/+/status", qos=0)
        c.loop_forever()

### Starter 4：`class DataProcessor(threading.Thread):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
class DataProcessor(threading.Thread):
    """TODO-3: 消费Queue → 去重(同source+ts跳过) → 振动>4.5告警 → 批量入库SQLite"""

    def __init__(self, q, batch=50):
        super().__init__(daemon=True, name="Proc")
        self.q, self.batch, self.buf = q, batch, []

    def run(self):
        db = sqlite3.connect("buffer.db", check_same_thread=False)
        db.execute("CREATE TABLE IF NOT EXISTS records("
                   "source TEXT, ts REAL, temp REAL, vib REAL)")
        seen = set()
        while True:
            try:
                d = self.q.get(timeout=5)
            except queue.Empty:
                self.flush(db)
                continue
            key = (d.get("source"), round(d.get("ts", 0), 1))
            # === 你的代码: 去重/告警/入buf; len(buf)>=batch时flush ===
            self.flush(db)

    def flush(self, db):
        if self.buf:
            db.executemany("INSERT INTO records VALUES(?,?,?,?)", self.buf)
            db.commit()
            self.buf.clear()

### Starter 5：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    q = queue.Queue(maxsize=10000)
    threads = [ModbusCollector(q), MQTTSubscriber(q), DataProcessor(q)]
    for t in threads:
        t.start()
    print("采集30秒后输出统计(Ctrl+C提前结束)...")
    try:
        time.sleep(30)
    except KeyboardInterrupt:
        pass
    threads[0].stop.set()
    s = threads[0].stat
    total = s["ok"] + s["fail"]
    print(f"采集率: {s['ok']}/{total} = {s['ok'] / max(total, 1):.1%} (目标≥95%)")

### Starter 6：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_P4.1.py` 或本notebook(含输出)
- ⚠ 线程命名：threading.Thread(name='Modbus')日志可辨
- ⚠ Queue.put(timeout=1)防永久阻塞
- ⚠ daemon=True保证Ctrl+C能退出

---

## 教师参考答案（来源：`教师参考/solutions.py` · `p41()`）

> **发放纪律：本附录勿随学生版下发。** 课堂按层放提示（D层演示照做/C层只给思路行/A层直接做）。

学生版 TODO 位置 → 下方参考实现的对应关系：

| 学生版 TODO 函数 | 参考实现说明 |
|---|---|

In [ ]:
def p41():
    """端到端最小参考(单源)——完整三线程见starter骨架补全"""
    import subprocess
    import sys
    import os
    import atexit
    import time as _t
    from pymodbus.client import ModbusTcpClient as Client
    base = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    mod_srv = os.path.join(os.path.dirname(base), "02-程序设计基础",
                           "虚拟产线", "mini_modbus_server.py")
    if os.path.exists(mod_srv):
        p_ = subprocess.Popen([sys.executable, mod_srv, "5020", "1"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        atexit.register(p_.terminate)
        _t.sleep(1.5)
    c = Client(host="127.0.0.1", port=5020)
    c.connect()
    rr = c.read_holding_registers(address=0, count=5)
    vals = rr.registers
    print("PLC1:", dict(zip(["temp", "press", "vib", "rpm", "status"], vals)))
    c.close()

## 运行与验证

上方参考实现可**直接运行**（合成数据、CPU 离线、结果可复现）。
教师可对照学生版逐 cell 讲解：学生填 TODO → 此处给出完成态。

> 多解法对比见同目录《多解法手册.md》与 `教师参考/solutions_multi*.py`。